# Kaggriculture: BC → DAgger → PSRO

This notebook deliberately starts from imitation learning, not PPO.

Pipeline:

1. freeze the current strong policy as a teacher;
2. collect independent teacher trajectories;
3. train structured behavior cloning;
4. add auxiliary win / future-income / opponent-market heads;
5. collect DAgger states under the student's own policy;
6. require an imitation gate before any RL;
7. build a PSRO-style empirical payoff table after the gate passes.

The primary evaluation target is match win rate.


In [ ]:

from __future__ import annotations

import copy
import json
import math
import random
import types
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 20260818
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FAST_MODE = True

TRAIN_GAMES = 8 if FAST_MODE else 40
VAL_GAMES = 4 if FAST_MODE else 16
DAGGER_GAMES = 4 if FAST_MODE else 20

BC_EPOCHS = 25 if FAST_MODE else 80
DAGGER_EPOCHS = 10 if FAST_MODE else 30

BATCH_SIZE = 128
LR = 2e-3
WEIGHT_DECAY = 1e-4

MAX_ACTORS = 17
MAX_MARKET_ORDERS = 10
MAX_QTY = 100

BC_GATE_FULL_ACTION = 0.85
BC_GATE_UNIT = 0.97
BC_GATE_MARKET_SLOT = 0.97

print("device:", DEVICE)
print("fast mode:", FAST_MODE)


In [ ]:
%%writefile teacher_agent.py
"""C166 anti-H4 meta counter.

C165 is preserved as the baseline. One addition only: after observing a
high-confidence near-mirror opponent executing the same four-step premium
front-run, the agent gets one step ahead of that policy (H5) on later matching
sales. The escalation is evidence-gated and skipped when town consumption on
the current step would immediately refill that product before the opponent acts.
"""
import base64
import copy
import json
import zlib

_TRACE = json.loads(zlib.decompress(base64.b85decode(
    'c-qxnU2j|05&SQD=7ag65BE*6nOcZq8M0g=H3%a>QxquDhqP}+{(EJKyyU$*J3G7Q&`O`e$mG4}`|Qrn&VK&y+24Qu?bly^JNwh;vk%uFKb~!8XaD&5Uw`}e;~ySB{`&K8|NP}YkDouEefR07FCXsj-hO)Xa5g*Jyjx#C{&&9G&OV>LySZPV1s}ft{QCXPkJq=4zkYMKdHk*U^T&7V^@p?BYX1)(*Xy^BfBtcOd-wiqc0T#@t_kOd*KB{Ejrw!*=7&!oj$SwP?DN@X{qf<cUBibr*SnSuKO34e9*B?Y+uOtIU*pCuabp*58}|b`YUTU8`-gW=jXHeV&654;xSf;t?7FEw`@2s!w{O2Z{_n%5r-Ah!Jo%>H`rGR_>%$<=**o^%1@rj$PwyXwVIRM1n9=XQ&H;PD^A|?z<N9HJ?>+DN19{%=n{X`lEq<Q0(Q|iSVk0rxqUdplrWdBuw*2rouvIdXXqrAej|@pTb+*AD?>_B@8jM6?>g*CbG<^Tq%FG?L&h)>}%nfJU@TN{PrwmD9=A;n`XV5lx=rtIK=Z5DN#J(wql{WoTZZwb1#L2K0)<LfOjq70Fy!CZ@`bj(OIJ6wSto!N5jZFb)c6bY>FtVrqNntOYqG8_A#eQ;kd%J$~@a6mU{lm@e&0n7;*50=%i>R`;Kr_e#_Etcuq27va7@cIZ_vc*C391~F@oy?)mEW#i-tui4c8Z^HKXK<C9nJf(i#Bx$jYoO8)erRYS=9Zvo_p+ebDnbq2lFaEF03zO9Yy>gr>Pg)gm)txXTia+Yk$iTez@y%XEAJ=(h&b#uQp{Eh47%=TQ#*&07Hleq0)}210wC1uz^SZU7<~ffvt><?={H%0n_D-c-TPZ`xef(VQ9)1VC}t5|65Mzv|-zFhm9e=`Q*<ZAMUR=->vWOfBK3FzT5?s@>RIQE#GNT3#D#q7vtnZw+nJsGtm+D+5c{Ay(RaK_R$*6*#`4%$wit0fVJ{Ae(#doCv5fMEW|_9?Ee5%B77$B!{}|@bWueeFz9V|guvt>=oUDA4a)=UpxCRVEp?Ios=;?y53z+}jfvr4ouKup1LE+I31n8ZLnrWH4HeW)3=Z1qmphLtN{$-Yf@&DEdSBHHIdXAH4=mqc#IK$jSZOy<dP~I@iH_gI$gv2mITLCRoH&(V7u@yzU)Jj0L>{vpXfJfr#T5r3t;*BEV>a?IfP-%8lI9w?J01Bh`D4tK?!6wjH!&86pJA`oM!VPbtHr-_8=c_czDv!TY&(OA%TFb7&N>Z;=Snu(0m{=F+?C90PGomD=)sEcs5%g!>aN1n9$93pm4d&HY==S|ADptMbC-ORq#ru#IAgtxjv5Qh8Xe(O5x!Sx80`vDhcR~LB-%|pfYt^$TzbQ8^8BH;#;1?#0bMwrf2TNl?)q_~#m56!XjudV^(3V9xxoqu0BfWC5Modqrb6!MB6C%%Q`5mLNFXBL6oy#lVF<<a2Cw-hmv?=pj!o_?2$P7{%RmRicN>6*31`^D(Z^)<`Q^5JH74#`hAJhP4ABr{7qb%AwKhu0r~r_DA@m?(O72>w0Y@3t?`6Cs><;@FJj3CG-rnDRa3+}8QisiQcXzu>>;Q-KklFEg>AW?^vh9Nf1<<RlTelG1x<zjH>c%AZ>W1eGD@K1Z@fHq`H89oWc26*(eRl#RcO=#syWEs%PaK<tkZr{pd0zUD*MykWl}h+pxVP(~h$uWgb_txzSt3Z{ZAa6}6~Sg+V=~B6?{Z6+$A+ZJ3<L!wNq#yhoPypP`D~5s3DH2BF^eEII!oOwWk(Wl3Y@;!!##;B*=006^XZt(FckpO2;mi-Tusv(0-YxsBSbmpnf(dIgf61wys*F!2x7_0qlMsULrOgG84^$Q3*62GSf<9lflITvQ2?a}EJka+S%bEbH{t??g7Lv@e03QR!<j`a!^60|RqPQqvE=al!_d0m25RXtz;KyKWTfKdPhX4TC;dj-ZUfMSVU4`K`7>@xtF1$FlieYnwWKOlZh@w^DIZLVYI#~yuT3N}q`}t9fXBV8`8PS#HWev7u;|eD3@VVbzkSz6@KmUe;Cf_>>ZwGH|KoFa7%>4GU2HofQAQ3)#`92;im<m~jTn6;oF`^mcPLz&CQbHB9-Ckeh6U0&ffm||!&|*uqd7Ql@4~5h(g=kg1@CaLGYMG=RTEgCuJ_oLJmJ_;la;HiA7Eeuqw(;*VbqP687~utzFRv6#z=ElEqfs}U7~}slK5b^GsP{8JWesY&1K$sXux3@;JU$7@J^?ZZz+zfi3YBVJ#!F(BTd5eKEBvigiZE-06<h)SP&x~r|FG7f?M9j6rHTjN@qEoUFeK<Ma0~)2KoF|Yt*5Ik;;6YY>Fbc>KE8j;6{BlGm%{Ik?~v$Ads*)wTw$9vK%}|ueLMZGgT8@_o0=np5yG;yxbN{Jk-V#WkVuGKz6*A|JFYZ)eeC**>z~YZB+~lGFk_vltn)?HiPuXuNDD)YB99YdB97688$;Zp$=d(lOI@JV3oDREs%wW^v)s=F5cq67Dv+?dX4VjBsE3=1aJQ&Cl>Dx(;shce-LaMbcrD96LeOZ91&}!96y@*E9G37R7%cac}Hhi3$Iu)<8+%`$vuvCQip71%_!~AKgIyM8nfI^ZI~+^^}3E^PB3+Eeo<mEjg_4*JAwNPC+4pE0XF6cVg>9=O+4<2T9EdJps|#z(v<53a4us6fDrgbaE8RM^+%h%9E}+#b#R!xu`AXxvEWG653>Dbu^zF@HQ$-~w7{abSlE$tvNj(qmZf`PQ^jhy{?Y2;-WxUWbw6%Z_T$jv5^Msf6;34ORam#Ir2~XlYI_9aUZKv?iJOc>MIm<PfT(s^Ym-B%HwmY4I>|7YK?JGMsM;P3Xew|-oa~i#oS(2WM4s_dCun~~D5Fkw4MAJT>XSzlJU_5C`HAyq8Z{#_^f4t(VLC8Bn?+!0NZQ3z!jO%ind~oXcWCl>B#W00|2V9D;ONsXP1pH_zm(6(4!G#Kpc#^rThP>e%nns)x#A;;uR&_ufIT`n0qHSjfd=Jl28+<1HI}gnI}!j?n$Mq<X;W7oyI$-`=$^yWCM|@#(1j1y&2R_+cRp$|!303z4JBSV94PvNW20r;1>sL0zhKX7(%bWnBp*Bqml83e*(3wB(K+1%-j~J?_dCd<>I&$jEVM%HkXfe>C*s{I>+%$6n4jAwz+u+F%IW~D5+q2}=qg6kh;5;!Fm$W6B{kS8G>rVs3te(IMW_Z-ogiM{4aPwTr53vP(}cR~JqMQ%jK3Q1kb*CXF>I#Jk^B#;G71+#A(rjt+NGBzkJ4ILugWe6J$ZuIQb~^-c{<5L;4m4{o#Zelib5@7+lFBTcGFOtJA-)JhFE4LGg|4m=9FsYi9w28r_@FPXJskNl8vTzM?kE-LY$zKZ3%rKhO|hnN=o!He3yhQ+!BvPeu*YBmi=JPn3}LEbCzjU72djnQb}L;q8mF#r}ut?fhe$5I;1}SAGw#b;VJz)%((^;E+lU!dY8l3jxYCx`!93d{k+okKA>nH;8Ifnl#()A3hT&;njR6e*pen*IJ;9$>@lNG%Op;f;L(y4h-WY1a9G23_>maofGb|oiU3ng-QtX`C+jF;4+oMr35P2|$0h>BB$FxPZ*pEyKivzus96Lm=9?)ANpXOU>VAZSM>l+C&)W5eDpB}eEJ(kgte2*BhxY*RHDG`lZW`u%Ka;sHnhRWkZ6ZuI8c;9y(y<T%p#tmNaTp4merz@}#~=?2!Y#tsK1X!&<4Zz!=upM923a9%rU*=rr1}%$^E0jqI}UT{26i18*%yF<>WLo1)5fz|*#j0s(@Nf0c8aS}u1oEh#%0>3ka(p?Wjw!D7B|(3E;$yO3hC~kTm7Nk*3-m5TM~m4@lK#es>R`Raxm(r(?>mJeNvn^(mSk_N<&U@s09)u3y|laaz<jUMK)FT*~X=wcP=hLG<{fFU#fEHqF>#C*k2=<u^80S#*_qX(3WlHpjK35g4|;gJ<lNcV#+LyzqmoEsx=NCm>^K*-fAC1IMbDP^2A_WlPEG_0sd8;cO)n4GngiS={XcI^1b$!q9j+{lr{}<;jA2VLjw1ao8sp~*`$`fI^MZl<cOdg<q9<@K!wpW{+G^SKIN4KZ6?ACwT`6y3uD1Pt<%-^)FHi=E-Li!a^0#$CT`ayN!pLwH5L5u9}7&?Ao;6MoI*J2yh@CmQ}j}W9c&{t;$7em0Qc^7d!6q{F<-9}&dj@dxiB%#ko?!wPB|v9fr#Hs4O=JdU$zBUIEjV_`^C2GQjZyHB(J|>s!33#hlP(Ic{@NEc61`dpp|oH341gFvtVBH{iB$$?L&at0IXFeaiP;I(j_}WK?mxV6xjSs2qoNPkaK&y*c-0Dw0DN(Z5w&Mq_zc<Dnk`w%o*17b{6PzC0!flXBqolv;;%6ut5ox33szLP6V+F0l+TGSYT8RZ?Gb=l&DG)HlMtsdB%+g+}MI#SP!j|7o}mcOvuR7Q|m>H6vM7Y&E+Ueg}Z4j<pOt;6{Q$+H7Su~CglurYGw~*se->sqh^kO+9+<7FZQl-M5%7ERO<rk#I3evSyx3)mdm46tUu}gETx4tm^$M6&(sNkR#7nbl`UcC6N_C@Lfv4I^w=Ckz;qV4wW(Yal@rGj2R+RvlFgJ8HDR34h`36~d>MR$30{#P7Y!ulf=djGM2akG#TucQT($H&qn1XhG^xUF!78Q$B?ezlgVXWD_;_Rn8=KcZiDP7Qfeb-$l|$|}HQcCko6DkHhqIv~KrK~vQfiGcp_6L+?@a89CJGSak!Y-;Q%LBd3W(q2kV*otG9(FauDnM$FM5Q)0$sd-zH%lRuR%t+5DRY!8TL_<fnMU${g+Ul8UE@VZ~n1D;Y_L+C6xj9(-ieXOVQlv6E_sXnS`My_|e}$$C-J6+2|%!IpWeT9G#J)p9w9!O!XBfti*0x&Dmn+)-iP$K*@0_5mEXg;pa<bR)OOK4MLU;snWw*vf)|nW-S{N#kH`jAiCbI6z)(xm1(fEJ<Fu(O6716h=F1`Ixgq!R~9gr{iKBwGz$wn&VuUYqD(wLqFQkah;QO!h5ONzzAP*nsjv>o0s}EZh#`B&J6@V`mMN1?Y6?X2SnFCiR<&gU&|Ex(;9Ak+*OW&ik`SOldbQ2;hi%p%(2dI~!LvuvaglPJm>f{BhXbKvya~MOo(*uaEzTk*yGB{sRGCf*cb}r(x9u--3NZhS5e*yi(>S_-Xes7PUW;92<Z$24SH}@tukLWBNG+j8w-JG~F!ji%dCnXaRu`!)7E>0oLQ~z-Wxp{GJXwHi1pZ(KL4SsLOV+GR6)$EsCL{i?vk(%RFH46g$es-xh79Mc=xu=nRl*0`)MCPt8n7VsL11r|%QiiS6VRx#-1H=0Dpk^ysRO16ff=MUT-|$YU3$Eb5%n=M6^!LMmjYtKY8Az#OIfDMp&?R{;<%WNV1g#0PU3k?@YdT#iJ8(Q16NgdQTK<|vOu$kF_4%|Ito3k1P2VfGMqYaQ^0<y#<Mbn!Jgc#1!@Qz!30F+s>aPSC}%KG2_8s9=wcd2N8Vh`A(oXWftRK4yz%)Y@R!58S;5}ecoo2VU~@5kp>+?&<EqS;^~a84rdFv*hTuUQqD%I8$a(XUfoBvTAdR}*R;oIWPOqz%WS@6p8N9F>Fss{ahyWD@TS%U5;Tdf@(YYn*9{Ff3x2l7%4;VUjl5#D|K8>w;Dxf2>R&WTWcc~JiCFn&cVNR?!7XVXrv@s8y`Levq=Lt2nDX1D|@Aehb(wY4ce;Zh}adF^NY(@rF_~u3!)2hgst+pkmOSs|iVvplk1^K~whQY=u&omjLv7`&5MsFlZ<4wrb=4wy9bT&4OVUUtF&Tv60>=j9|j4=|~<TpqqNGTvx)LNxE@gV9LJYLnXi4YOMSXp3xk~*N3!Gth?q#Nmlc8Q~PwXK!BbwEX5)IU9R1qfM-hBCEmdw|*E_J|;3;_=E1c(^VjWJndDX!e0rFsT;*fU4fMfNrx2@n{9VetA`~RJEnvc?a!rWna^)lvcnFCZCM&<O=8!;$HP_6@(*|xt@-8S45#VX?~^D%&vI>QO++@@(Jz{CNr8#IF+L0x`GuN`F)K^>df=(;kthTQyXzEfS%4k(v6YNw;upyK9cHYG<hJZyXp;u*S%AzqVWhkNyisUq}CErLl<d0e0r49SHlWaO~P5A5@O7tBQnhb;TxY(j)v}}s>bVWs%ofhx?ZV7%dK*2<8~6>i)y85c{4N%iR1_INilhlFp0IX%qzRi0|gK%lZ@1v@p9v=15&vJ!C6esF0GDUMu=Ld5rKJuunH4JJDnJ?mSGXom=zZ>UJ|<1LK7(h+Dy&J3c`6s4im6(y}>Em@pKXhVhe8&yt~CB-5qy?N`{p-x65!;ap3y0XQ)_Y2Ra!E;GnKhgd)eRuuYjRpj(8XpYTLu4uM5Yo}r@nT)fy;3U7^QBCm{)-$fdT0mSA2q$$-ssAeWKYFMl;{&?#498fwh3RBcV)v}&op0W~`cP(z`gPM_0ER|ned0ll$49MYu>3%hF>PUW<95Cdatr<bid7sEI3!|8U=gO{dNgNQ*<@)ySJ*Z8)QMg}$VU(uJX!c5za0SUiwVMh`oKZb@F(#dw3Q5#LDyqh^pB>l8u{nEivWlQIV!O0zzJNa+c(Z2EzzQH%(QaDji$pDNQDjG}4##5clOrY}3YCD7gd5Vk!5~GFRAR<QGE1xQcAZw`!7AE;u$Abtvx7zGfh`u#x)6{A7~ftLuXNn{woDR&5;m0%9p*7yXPkxWBpHOOEn-`1%SE!8s4!zT|IO>qv?ZKQ<w2=DFE9u)#?o4)r6^MW;MbUaGj1BTtWqmZpxXceO>)^m=7<de1OhMI_sU2`GeJ}sxmE>;+3o9C+BP$2laJyBmm*OX5wT#PQZfzbC^=HBzN+P9S(1?+|K-V_Vj<4m^>}X3q}SU-$GDn`?SGh<HJVyKw~i>l*i#rNszBUm(G1Sbua1XvH^i$>JgHjgQN~xS18y-;FQ$IIx!df^mLP%i)727Ls71D%*s5Zxz~*zPsTWbJ5BCO`?NNFQ|7OhGY8A4Gij+{{KGHlA$DJxDcNV&U{eXujmte(Tc;ZNWyRs&Nh-S>Ym(}6nuH@!RA}5MJs8;Di%Hs@Pyq4rz8Mxo6c6Z=cvz{GQE!&ZJm=W$0d``4AgCY<?+?LxoRsNfl%o*hU5?Z9?fa0YTohR17|L)Vx?b|Pp_Q}Ji`&j#rr|FOmpu}?dAOSY;<B#90M;1&<=Eu4Opr!jkFXe3Q-hhr0a?inFXn;qOQCC>OpaMazP~AK-3V22*`d1FJbgAh}knX_ejWoB#QauvbLo%HqR2YRH0O+w_<zrjLaIP~lny;+WC#dhi=@-gJsRw#o<SJR?aaSs6RSr7iRW(eem?jy~le$V21Tp=MW-1d^Vj@o*ryX-dY@;d_f|5vkY|5)%mTITfztOWOIQ&GoUL+hi(Hon=UA=0?XO^$NU8&bo@}f>rpl8`Xvo3`N6e&y`PKoL0KJ-$|@gXuKBAimunwWRs9kHS}5tAsPog&kS^jf64Os&t}56`8mt)6*D!`z_{-lo_w3)u*EKXJpC>F7uxNNp=2K{*p)3+fY)6rk#cK_jFq4=cqg8bp5NEI2ReBT<ttS$?wwEM_Ylc;<SpAxaIipd3g=>yZ|=AbD6p14ucV+^|SXNzbxk^T^6une$D_Xj3JhSaKTy3K$OabOc4BPYEoV7UfAcPNJmXYl>(gi%w#0E3oT;-wVX8s+2NHl`EBtkbz8?+>n`d##B{E4Q%3RhPEA9PNLQda!8qkOl2i0I;gB2h0-ThY&b6vyLw6IwYl6#B|*uNeCp-co>v`P__PbP1Z&J*K4~QrW=N60XpVAL>j+smLB$g<q)mC$d`SK%g;LZqa1mFmskRClH2oT*t~UCb2_7aY>YQ&Y;d6+1D5kAyLo!Bd!rPnjRA&vLc<ilE@fqgqI{`s8bbaDvm?%XI$LEt$G<La)p=F7B*r$M|Q(G@`Ad*#FV;c;F&xwn@A)Ho`KO`zMY}VZiP<qa=&bX$@B_iBaQFILhw(#U|5?Uw>!KH~r&6|wUf@Kqld=kcfXR2t)?JtBqodqVB4GnNBYDSR^nnWEkv}<$`;|)l%s3Zc@V>CJ&Mmnfh6+#Eo{o@u2;zQWSk^`W8w50%KhTcmlKh#kvo(=wPq7dLw9-~v5HdsT!!c$VwjqYe|5<xDGfHVg67OkAHU9N7`sGPar?D~NmYA`E>Knj}8?2BoFc@@bg#tg_|m&rx8GvFVm-^u}>im0>2hd?uk#<FdyTJ$_F=yfs?V+BehC4Aembef7T`RLfPScHkDH7fZk#hObQ{)Z?A(}KoDScjr$6LqP3&Wu)RDWy%)A5_D3YZP)pu9&EgQ)JyU+a|hjWhGsAvt=w-US^5grJP62eR7d`v)t;WWUhbfo(HU%S+PjD&=Qn|m~zF4mKbrFTk)r8YXUoq31+~vS8;(@NbObZfO(oOv1FkP(lE_QF`M)}L(rmchKpW}Ds&=ip-*--hnk}4U8nN5d@MA~;ew{MPwG!@!lhF9B|=ON$rQX`8^_>$_12;S!bV8yPiq6wdR(KkV&!OIuQE!~o)T}3Du;ITRHaE#15MNmb~$Mo`A4Ei)mot~kn&|>dLcx~2xN8?t=NJcsQpeGCe3~jw6vuLzRn1PixL_n)DJj8Uh-{>S-t92ElO!HO~7Wh#3Dz}sgPO5NM6UP%_o-AZOkiKc~Gvxwpm7ml_q2%Su|NB(@=V+-Z&|kA5dk)*10C^2?#MP+XVJOwqBFPw&sa+4n*To8UGZiV#Zf<LR5n1B9b5MHPEXvlFzdbIMN$EzEX&|oI7gi9Xx1FmtKsJixQ_Yl70#LEk~cy`CS4Hi|~|aCGUjvTs9$v7Ft=0QRWPb1{h9c1BDQW64j3lz>GA~T*4iL0_lEnr2xx60ZoI@-Q`+Vn)}OqpEeE#M{Psok`Fo)1DC}dmhlNgVW=o6pQ#T~qGK_H&B>$Stjp%b={3SG>imxlMeo3wSQ55P@6a%L0G>I4f)}IMlOvMrGUpal!%@SSE$m#UggBDb=v1AXxf3Rqej>O5*dcnMQ&vTcDX>z!F%}g@`A~{1N#1Z2r$|sjuGLFQX=0?IL<XFd1wTSOkmN(1EJLathv}y8u?(3Wl<BD5POsy*m`v(O)+l1n4cooicGe5g272Irt_z}4r<D&8*{<w1(mS)<zMSHSG*KjHs7N)@80$1YI}){QM5TyP#Y4wopqN5X1r<2jE{jz<vh==lP57u;(uP>z_q5U`Uh;{GSTRuQnXjajq0uQek;KVHm_r4JDnd;h#nLo00?f43z_11kFTY0*KeU}oU<RERdo4}Wfua+}TM&&TC3<-0?jP=PZWQimf)2#R89`^b7^$&eSiPkRoEz1baRhu*72^`;P$KWQmh?d#_=q4x3rkX@tPt8Hpk|N~!8nvvB^9N8;f(XlEhe2mPHL&OQ-fgrcpeyaQiK$xEtJY1!(<V_kCOT#=*DpAmZVR|+@4HZvT7?8VVQlk+L<5rd?|I*Bx(Yhf7vZtMiM*>7%%SkmOikeCwQpYF{CP6QY9q-LPi^2ke(E|G^Ick&2VlYx2T#d0UbM#;)vS`Y#db{n;T^;52&~%#k$A1c6SkuX~i#?uH;6q7^BQb4~l!F#X=no5&w~NEJ+>)=bx#pAL3%>g0gB6ZdzO#?3R{Qhs|3YCX!*uDvsG>T~cYOOeRf`Ax9vjoFC3p9cf}5B_OB?ZwlZQuaZVde=6dcyOcJn20M)oz)hbz0Bjg<qX_o`1i&V_D~c!~>zvgj35<zwCr1ggpb{yE)b|p9A_{E>KLGhzgdU_Q*rGrwVMW_K=4~qVE|}-N2_qHrOj1gi#YS)yTdh4MgERM>NI{;?HijfSSg8my34-#mKblHw2&yc##^fMv8)y!zi$0Eg7lGI|vP0Fhppn<0sF~}b&)pw4v55m84O`uc57)y+W8O-_38!4kV*T=spw*Do_naPi0+B6jaa4Z|SEgll5jEgy<<vQ)J(0bmu@76PoTVlsOV$6!TtqQYiHHnlDd#6QHmTxgXjhb6K`JLACOM)Im2}V*AQnAPNM!pIb)cXfQ)jW6;DcPvY2G-_is7es4mXjjXRc+^cV?qTw;3sUGULY!z@iFN+Vf0WK4o|C1bi`eTTIONgVx^65?z5f%Uy671|!pQjp*qMgfbPWg?$RaOe+XxT?$4mr;$!rTb0z|XpUJ@&@&K2@vG;g{I5m^xvM{vC%!9mZ}LH@o<Tz9yX|n1SYCx!i>(UoDAs}VW#{m6=q=_-Du+gcv(tDRRQQu!Lbe^Q)>F=!<%ln7h)^0i8xW)xpK5|BYAPw#TA)FbQT$b|zLA*E!rd>$pg$cbKJ!8K<}r?awaw&GH;M77WhN_4j2LDKQaX*MRynPF2T;BxTh?`Ui6X5ihME(N%Haj*6?z3$WMHS77BfHMZ4h4vMIe&ph_n^}O^zHBUW?ODW27a8y1->0p?cBC(cjP?rxqMq?@WQ8eM<*X07O7d21+HifwxQ~#j;Gr#;KaNAsRJtGV}Vjw80=wOC;#Px2M13*QODISt#R68glUA%|8K_&6IykV`vMZ`1S5()VCv{Mj~N<6mJThyJg0^D^z17wKaKqoM?eElJWNU?f(F(%jUT'
)).decode("utf-8"))

_SELLABLE = ("STRAWBERRY", "MELON", "MILK", "WOOL", "EGG", "TOMATO", "CARROT", "WHEAT", "FERTILIZER")

_FRONT_RUN_HORIZON = 4
_FRONT_RUN_ITEMS = ("MELON", "STRAWBERRY", "MILK", "WOOL")
_BASE_PRICE = {"MELON": 250, "STRAWBERRY": 120, "MILK": 160, "WOOL": 200}
_GLUT_WEIGHT = {"MELON": 3.5, "STRAWBERRY": 2.0, "MILK": 2.0, "WOOL": 3.2}
_LAST_STEP = -1
_CLONE_CONFIDENCE = 0

# --- H5 meta layer ---------------------------------------------------------
# Disabled until public flow confirms an H4 mirror.
_H4_META_ACTIVE = False
_H4_META_EVIDENCE = 0
_PREV_MARKET_INV = None
_PREV_TOWN_SHOPS = ()
_PREV_ACTION = None
_PREV_SHED = None
_PREV_PRICES = None
_PREV_STEP = -1

_SHOP_PRODUCTS = {
    "BAKERY": ("EGG", "WHEAT"),
    "PIZZA_SHOP": ("MILK", "TOMATO", "WHEAT"),
    "BRUNCH_SPOT": ("EGG", "WHEAT", "STRAWBERRY"),
    "YARN_STORE": ("WOOL",),
    "ICE_CREAM_SHOP": ("STRAWBERRY", "MILK", "WHEAT"),
    "PET_CAFE": ("CARROT",),
    "SMOOTHIE_SHOP": ("STRAWBERRY", "MILK"),
    "FARMERS_MARKET": ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY"),
}


def _public_signature(farm):
    """Compact public fingerprint for detecting a mirrored build."""
    counts = {item: 0 for item in (
        "COW", "SHEEP", "GOOSE", "WHEAT", "CARROT", "TOMATO",
        "STRAWBERRY", "MELON", "PASTURE", "COOP", "WEED",
    )}
    for row in farm.get("tiles", []) or []:
        for tile in row or []:
            if not isinstance(tile, dict):
                continue
            for key in ("animal", "crop", "kind"):
                value = tile.get(key)
                if value in counts:
                    counts[value] += 1
                    break
    positions = [farm.get("farmer", [0, 0]), *(farm.get("hands", []) or [])]
    return (
        len(farm.get("hands", []) or []),
        tuple(sorted(farm.get("unlocked_quadrants", []) or [])),
        tuple(sorted(tuple(position) for position in positions)),
        tuple(counts[item] for item in sorted(counts)),
    )


def _signature_distance(left, right):
    distance = abs(left[0] - right[0])
    distance += 3 * abs(len(left[1]) - len(right[1]))
    distance += sum(abs(a - b) for a, b in zip(left[3], right[3]))
    if left[2] != right[2]:
        distance += 2
    return distance


def _update_clone_profile(obs, step):
    global _CLONE_CONFIDENCE
    if step not in (4, 24) and not (step >= 48 and step % 24 == 0):
        return
    farms = obs.get("farms", []) or []
    if len(farms) < 2:
        return
    player = int(obs.get("player", 0) or 0)
    distance = _signature_distance(
        _public_signature(farms[player]),
        _public_signature(farms[1 - player]),
    )
    if distance <= 1:
        _CLONE_CONFIDENCE = min(8, _CLONE_CONFIDENCE + 1)
    elif distance <= 4:
        _CLONE_CONFIDENCE = max(0, _CLONE_CONFIDENCE - 1)
    else:
        _CLONE_CONFIDENCE = max(0, _CLONE_CONFIDENCE - 3)



def _sell_qty(action, item):
    total = 0
    for order in (action or {}).get("market", []) or []:
        if (
            isinstance(order, list) and len(order) >= 3
            and order[0] == "SELL" and order[1] == item
        ):
            total += max(0, int(order[2] or 0))
    return total


def _trace_sell_qty(step, item):
    if not (0 <= step < len(_TRACE)):
        return 0
    total = 0
    for order in _TRACE[step].get("market", []) or []:
        if (
            isinstance(order, list) and len(order) >= 3
            and order[0] == "SELL" and order[1] == item
        ):
            total += max(0, int(order[2] or 0))
    return total


def _town_demand(step, shops, item):
    """Exact default-configuration demand applied AFTER market on `step`."""
    demand = 0
    if step % 4 == 0:
        for shop_name in shops or ():
            products = _SHOP_PRODUCTS.get(shop_name, ())
            if item in products:
                demand += 2 if len(products) == 1 else 1
    if step % 24 == 0 and item != "FERTILIZER":
        demand += 1
    return demand


def _remember_market(obs, step, action):
    global _PREV_MARKET_INV, _PREV_TOWN_SHOPS, _PREV_ACTION
    global _PREV_SHED, _PREV_PRICES, _PREV_STEP
    market = obs.get("market") or {}
    _PREV_MARKET_INV = dict(market.get("inventory") or {})
    _PREV_PRICES = dict(market.get("prices") or {})
    _PREV_TOWN_SHOPS = tuple((obs.get("town") or {}).get("unlocked_shops", []) or [])
    _PREV_ACTION = copy.deepcopy(action)
    _PREV_SHED = dict(((obs.get("private") or {}).get("shed") or {}))
    _PREV_STEP = step


def _observe_h4_meta(obs, step):
    """Confirm that the opponent is itself executing C165-style H4 front-runs.

    Premium products cannot be bought from the market, so their public inventory
    delta decomposes cleanly into our sales + opponent sales - town demand.
    A signal is accepted only when:
      * farms are already strongly near-mirror,
      * our previous action really performed an H4 pre-sale,
      * the opponent supplied a comparable amount on the same turn,
      * no base-tape sale of that item explains the timing.
    """
    global _H4_META_ACTIVE, _H4_META_EVIDENCE
    if (
        _H4_META_ACTIVE
        or _PREV_MARKET_INV is None
        or _PREV_ACTION is None
        or _PREV_SHED is None
        or _PREV_STEP != step - 1
        or _CLONE_CONFIDENCE < 3
    ):
        return

    market = obs.get("market") or {}
    cur_inv = market.get("inventory") or {}
    cur_prices = market.get("prices") or {}

    for item in _FRONT_RUN_ITEMS:
        # $1 floor: inventory no longer identifies sold volume.
        # Ignore that observation.
        if float((_PREV_PRICES or {}).get(item, 2) or 0) <= 1:
            continue
        if float(cur_prices.get(item, 2) or 0) <= 1:
            continue

        target = _PREV_STEP + 4
        target_qty = _trace_sell_qty(target, item)
        if target_qty <= 0:
            continue

        # Accept only a clean H4 timing match.
        # Normal / nearer replay sales are noise here.
        if _trace_sell_qty(_PREV_STEP, item) > 0:
            continue
        if any(_trace_sell_qty(s, item) > 0 for s in range(_PREV_STEP + 1, target)):
            continue

        own_requested = _sell_qty(_PREV_ACTION, item)
        own_supply = min(
            max(0, int((_PREV_SHED or {}).get(item, 0) or 0)),
            own_requested,
        )
        if own_supply < 2:
            continue

        demand = _town_demand(_PREV_STEP, _PREV_TOWN_SHOPS, item)
        observed_delta = int(cur_inv.get(item, 0) or 0) - int(_PREV_MARKET_INV.get(item, 0) or 0)
        opp_supply = observed_delta + demand - own_supply

        # Keep mild quantity drift; reject unrelated dumps.
        if opp_supply >= 2 and 0.40 <= (opp_supply / max(1, own_supply)) <= 2.50:
            _H4_META_EVIDENCE += 1
            _H4_META_ACTIVE = True
            return


def _h5_meta_counter(action, obs, step):
    """One-step second-order preemption against a confirmed H4 opponent.

    If their C165-style policy is expected to move a base sale from t to t-4,
    sell at t-5 instead. Skip the attempt when town demand after our market phase
    would refill the item before the opponent acts on the next step.
    """
    if not _H4_META_ACTIVE:
        return False
    target = step + 5
    if target >= len(_TRACE):
        return False

    orders = list(action.get("market", []) or [])
    if len(orders) >= 10:
        return False

    already = {}
    for order in orders:
        if isinstance(order, list) and len(order) >= 3 and order[0] == "SELL":
            already[order[1]] = already.get(order[1], 0) + max(0, int(order[2] or 0))

    shed = (obs.get("private") or {}).get("shed") or {}
    shops = tuple((obs.get("town") or {}).get("unlocked_shops", []) or [])
    prices = ((obs.get("market") or {}).get("prices") or {})

    choices = []
    for item in _FRONT_RUN_ITEMS:
        planned = _trace_sell_qty(target, item)
        if planned <= 0:
            continue
        # Town demand can reset the price edge.
        # Skip H5 when that happens.
        if _town_demand(step, shops, item) > 0:
            continue
        available = max(0, int(shed.get(item, 0) or 0) - already.get(item, 0))
        quantity = min(available, planned)
        if quantity <= 0:
            continue
        price = float(prices.get(item, _BASE_PRICE[item]) or 0)
        priority = price * quantity * _GLUT_WEIGHT[item]
        choices.append((priority, item, quantity))

    if not choices:
        return False

    _, item, quantity = max(choices)
    # Put the pre-sale first in the market queue.
    action["market"] = [["SELL", item, quantity], *orders][:10]
    return True


def _front_run(action, obs, step):
    """Sell one premium line immediately before a clone's expected glut."""
    if _h5_meta_counter(action, obs, step):
        return
    if _CLONE_CONFIDENCE < 1 or _FRONT_RUN_HORIZON <= 0:
        return
    orders = list(action.get("market", []) or [])
    if len(orders) >= 10:
        return
    already = {}
    for order in orders:
        if isinstance(order, list) and len(order) >= 3 and order[0] == "SELL":
            already[order[1]] = already.get(order[1], 0) + max(0, int(order[2] or 0))
    planned = {}
    end = min(len(_TRACE), step + _FRONT_RUN_HORIZON + 1)
    for future_step in range(step + 1, end):
        distance = future_step - step
        for order in _TRACE[future_step].get("market", []) or []:
            if not (
                isinstance(order, list) and len(order) >= 3
                and order[0] == "SELL" and order[1] in _FRONT_RUN_ITEMS
            ):
                continue
            item = order[1]
            quantity = max(0, int(order[2] or 0))
            if item not in planned:
                planned[item] = [distance, quantity]
            else:
                planned[item][1] += quantity
    shed = (obs.get("private") or {}).get("shed") or {}
    prices = ((obs.get("market") or {}).get("prices") or {})
    choices = []
    for item, (distance, quantity) in planned.items():
        available = max(0, int(shed.get(item, 0) or 0) - already.get(item, 0))
        quantity = min(available, quantity)
        if quantity <= 0:
            continue
        price = float(prices.get(item, _BASE_PRICE[item]) or 0)
        priority = (
            price * quantity * _GLUT_WEIGHT[item]
            + (_FRONT_RUN_HORIZON + 1 - distance) * _BASE_PRICE[item]
        )
        choices.append((priority, item, quantity))
    if choices:
        _, item, quantity = max(choices)
        orders.append(["SELL", item, quantity])
        action["market"] = orders[:10]


def _terminal_liquidation(action, obs, step):
    """Replay-derived safety net: leave no sellable shed inventory at season end."""
    if step < 680:
        return
    shed = (obs.get("private") or {}).get("shed") or {}
    market = action.setdefault("market", [])
    already = {
        order[1]
        for order in market
        if isinstance(order, list) and len(order) >= 2 and order[0] == "SELL"
    }
    for item in _SELLABLE:
        qty = int(shed.get(item, 0) or 0)
        if qty > 0 and item not in already and len(market) < 10:
            market.append(["SELL", item, qty])


def _shed_access(size):
    half = size // 2
    return [(half - 1, half - 1), (half, half - 1), (half - 1, half), (half, half)]


def _move_toward(pos, target, tiles):
    x, y = pos
    tx, ty = target
    choices = []
    if tx < x:
        choices.append(("WEST", (x - 1, y)))
    if tx > x:
        choices.append(("EAST", (x + 1, y)))
    if ty < y:
        choices.append(("NORTH", (x, y - 1)))
    if ty > y:
        choices.append(("SOUTH", (x, y + 1)))
    size = len(tiles)
    for op, (nx, ny) in choices:
        if 0 <= nx < size and 0 <= ny < size and tiles[ny][nx] != "LOCKED":
            return [op]
    return ["PASS"]


def _terminal_action(obs):
    """Observation-driven final-eight-turn harvest/drop/sell controller."""
    player = int(obs.get("player", 0) or 0)
    farm = (obs.get("farms") or [])[player]
    private = obs.get("private") or {}
    tiles = farm.get("tiles") or []
    size = len(tiles)
    positions = [farm.get("farmer", [0, 0]), *(farm.get("hands") or [])]
    inventories = list(private.get("inventories") or [])
    inventories.extend({} for _ in range(len(positions) - len(inventories)))
    sheds = set(_shed_access(size))

    available = {
        (x, y)
        for y, row in enumerate(tiles)
        for x, tile in enumerate(row)
        if isinstance(tile, dict) and int(tile.get("yield_units", 0) or 0) > 0
    }
    actions = []
    pending = {}
    for pos_raw, inventory in zip(positions, inventories):
        pos = tuple(pos_raw)
        inventory = inventory or {}
        load = sum(max(0, int(v or 0)) for v in inventory.values())
        x, y = pos
        tile = tiles[y][x] if 0 <= y < size and 0 <= x < size else None
        if load > 0 and pos in sheds:
            action = ["DROP"]
            for item, count in inventory.items():
                if item in _SELLABLE:
                    pending[item] = pending.get(item, 0) + max(0, int(count or 0))
        elif isinstance(tile, dict) and int(tile.get("yield_units", 0) or 0) > 0:
            action = ["HARVEST"]
            available.discard(pos)
        elif load > 0:
            target = min(sheds, key=lambda q: abs(q[0] - x) + abs(q[1] - y))
            action = _move_toward(pos, target, tiles)
        elif available:
            target = min(available, key=lambda q: (abs(q[0] - x) + abs(q[1] - y), q[1], q[0]))
            available.discard(target)
            action = _move_toward(pos, target, tiles)
        elif isinstance(tile, dict) and tile.get("fertilizer_available", False):
            action = ["COLLECT_FERTILIZER"]
        else:
            action = ["PASS"]
        actions.append(action)

    shed = dict(private.get("shed") or {})
    for item, count in pending.items():
        shed[item] = int(shed.get(item, 0) or 0) + count
    prices = ((obs.get("market") or {}).get("prices") or {})
    sells = [
        (int(shed.get(item, 0) or 0) * int(prices.get(item, 1) or 1), item, int(shed.get(item, 0) or 0))
        for item in _SELLABLE
    ]
    sells = [row for row in sells if row[2] > 0]
    sells.sort(reverse=True)
    market = [["SELL", item, qty] for _, item, qty in sells[:10]]
    if int(obs.get("hour", 0) or 0) <= 1:
        already = int(farm.get("hires_today", 0) or 0)
        for _ in range(min(10 - len(market), max(0, 8 - already))):
            market.append(["HIRE"])
    return {"farmer": actions[0], "hands": actions[1:], "market": market[:10]}


def agent(obs, config=None):
    global _LAST_STEP, _CLONE_CONFIDENCE
    global _H4_META_ACTIVE, _H4_META_EVIDENCE
    global _PREV_MARKET_INV, _PREV_TOWN_SHOPS, _PREV_ACTION
    global _PREV_SHED, _PREV_PRICES, _PREV_STEP

    step = min(int(obs.get("step", 0) or 0), len(_TRACE) - 1)
    if step == 0 or step <= _LAST_STEP:
        _CLONE_CONFIDENCE = 0
        _H4_META_ACTIVE = False
        _H4_META_EVIDENCE = 0
        _PREV_MARKET_INV = None
        _PREV_TOWN_SHOPS = ()
        _PREV_ACTION = None
        _PREV_SHED = None
        _PREV_PRICES = None
        _PREV_STEP = -1

    _update_clone_profile(obs, step)
    _observe_h4_meta(obs, step)
    _LAST_STEP = step

    if step >= 717:
        action = _terminal_action(obs)
        _remember_market(obs, step, action)
        return action

    action = copy.deepcopy(_TRACE[step])
    _front_run(action, obs, step)
    _terminal_liquidation(action, obs, step)
    _remember_market(obs, step, action)
    return action


In [ ]:

TEACHER_PATH = Path("teacher_agent.py").resolve()


def make_teacher():
    """Create an isolated teacher with its own stateful module globals."""
    module = types.ModuleType(f"teacher_{random.randrange(1 << 30)}")
    source = TEACHER_PATH.read_text(encoding="utf-8")
    exec(compile(source, str(TEACHER_PATH), "exec"), module.__dict__)
    return module.agent


def to_plain(value):
    """Convert Kaggle Struct-like observations to plain Python containers."""
    return json.loads(json.dumps(value))


def audit_replay(path):
    """Observation at replay step t must reproduce action stored at t+1."""
    replay = json.loads(Path(path).read_text(encoding="utf-8"))
    result = {}

    for seat in (0, 1):
        teacher = make_teacher()
        exact = 0
        total = max(0, len(replay["steps"]) - 1)

        for step in range(total):
            obs = copy.deepcopy(replay["steps"][step][seat]["observation"])
            obs["step"] = step

            predicted = teacher(obs)
            recorded = replay["steps"][step + 1][seat]["action"]
            exact += int(predicted == recorded)

        result[seat] = exact / max(1, total)

    return result


# Optional audit if the replay is attached to the notebook session.
audit_candidates = [
    Path("/kaggle/input/92401577/92401577(1).json"),
    Path("/kaggle/working/92401577(1).json"),
    Path("92401577(1).json"),
]

for candidate in audit_candidates:
    if candidate.exists():
        print("replay audit:", candidate, audit_replay(candidate))
        break


In [ ]:

try:
    from kaggle_environments import make
    HAVE_ENV = True
except ModuleNotFoundError:
    HAVE_ENV = False

print("kaggle_environments available:", HAVE_ENV)


def episode_to_records(replay, teacher_seats, episode_id):
    steps = replay["steps"]
    final_rewards = replay.get("rewards") or [
        steps[-1][0].get("reward", 0),
        steps[-1][1].get("reward", 0),
    ]

    records = []

    for seat in teacher_seats:
        opponent = 1 - seat
        win = float(final_rewards[seat] > final_rewards[opponent])
        margin = float(final_rewards[seat] - final_rewards[opponent]) / 10000.0

        for step in range(len(steps) - 1):
            obs = copy.deepcopy(steps[step][seat]["observation"])
            obs["step"] = step

            action = copy.deepcopy(steps[step + 1][seat]["action"])
            opponent_action = copy.deepcopy(
                steps[step + 1][opponent]["action"]
            )

            current_money = float(
                (obs.get("farms") or [{}, {}])[seat].get("money", 0) or 0
            )

            future_index = min(len(steps) - 1, step + 24)
            future_obs = steps[future_index][seat]["observation"]
            future_money = float(
                (future_obs.get("farms") or [{}, {}])[seat].get("money", 0) or 0
            )

            records.append(
                {
                    "episode_id": episode_id,
                    "seat": seat,
                    "step": step,
                    "obs": obs,
                    "action": action,
                    "opponent_action": opponent_action,
                    "win": win,
                    "margin": margin,
                    "next_day_income": (future_money - current_money) / 10000.0,
                }
            )

    return records


def run_teacher_episode(seed, opponent_kind, teacher_seat):
    if not HAVE_ENV:
        raise RuntimeError("kaggle_environments is not available.")

    teacher = make_teacher()

    if opponent_kind == "teacher":
        opponent = make_teacher()
    elif opponent_kind == "random":
        opponent = "random"
    else:
        raise ValueError(f"unsupported opponent: {opponent_kind}")

    agents = [None, None]
    agents[teacher_seat] = teacher
    agents[1 - teacher_seat] = opponent

    env = make(
        "kaggriculture",
        configuration={"episodeSteps": 720, "seed": int(seed)},
        debug=False,
    )
    env.run(agents)
    replay = env.toJSON()

    teacher_seats = [teacher_seat]
    if opponent_kind == "teacher":
        teacher_seats = [0, 1]

    episode_id = f"{opponent_kind}-seat{teacher_seat}-seed{seed}"
    return episode_to_records(replay, teacher_seats, episode_id), replay


def collect_teacher_dataset(num_games, seed_offset):
    if not HAVE_ENV:
        return []

    records = []

    for game_index in range(num_games):
        opponent_kind = "teacher" if game_index % 2 == 0 else "random"
        teacher_seat = game_index % 2
        seed = seed_offset + game_index

        episode_records, replay = run_teacher_episode(
            seed=seed,
            opponent_kind=opponent_kind,
            teacher_seat=teacher_seat,
        )
        records.extend(episode_records)

        print(
            f"{game_index + 1:02d}/{num_games}",
            opponent_kind,
            f"seat={teacher_seat}",
            f"transitions={len(episode_records)}",
            f"rewards={replay.get('rewards')}",
        )

    return records


train_records = collect_teacher_dataset(TRAIN_GAMES, 1000)
val_records = collect_teacher_dataset(VAL_GAMES, 9000)

print("train transitions:", len(train_records))
print("validation transitions:", len(val_records))


In [ ]:

PRODUCTS = [
    "WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON",
    "EGG", "MILK", "WOOL", "FERTILIZER",
]
CROPS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON"]
ANIMALS = ["GOOSE", "COW", "SHEEP"]
ALL_ITEMS = PRODUCTS + ANIMALS

SHOP_NAMES = [
    "BAKERY", "PIZZA_SHOP", "BRUNCH_SPOT", "YARN_STORE",
    "ICE_CREAM_SHOP", "PET_CAFE", "SMOOTHIE_SHOP", "FARMERS_MARKET",
]

BASE_PRICE = {
    "WHEAT": 25,
    "CARROT": 35,
    "TOMATO": 60,
    "STRAWBERRY": 120,
    "MELON": 250,
    "EGG": 50,
    "MILK": 160,
    "WOOL": 200,
    "FERTILIZER": 100,
}

UNIT_OPS = [
    "PASS", "NORTH", "SOUTH", "EAST", "WEST",
    "PICKUP", "DROP", "PLANT", "WATER", "HARVEST", "FERTILIZE",
    "BUILD_COOP", "BUILD_PASTURE", "DIG", "PLACE",
    "FEED", "COLLECT_FERTILIZER", "CARE",
]

MARKET_OPS = [
    "PAD", "BUY_SEED", "BUY_PRODUCT", "BUY_ANIMAL",
    "SELL", "HIRE", "BUY_LAND",
]

ITEMS = ["NONE"] + ALL_ITEMS

UNIT_OP_TO_ID = {value: index for index, value in enumerate(UNIT_OPS)}
MARKET_OP_TO_ID = {value: index for index, value in enumerate(MARKET_OPS)}
ITEM_TO_ID = {value: index for index, value in enumerate(ITEMS)}

ID_TO_UNIT_OP = {index: value for value, index in UNIT_OP_TO_ID.items()}
ID_TO_MARKET_OP = {index: value for value, index in MARKET_OP_TO_ID.items()}
ID_TO_ITEM = {index: value for value, index in ITEM_TO_ID.items()}


def farm_summary(farm):
    counts = {
        key: 0
        for key in CROPS + ANIMALS + ["WEED", "PASTURE", "COOP"]
    }
    yields = {key: 0 for key in PRODUCTS}

    for row in farm.get("tiles", []) or []:
        for tile in row or []:
            if not isinstance(tile, dict):
                continue

            kind = tile.get("kind")

            if kind == "PLANT":
                crop = tile.get("crop")
                if crop in counts:
                    counts[crop] += 1
                if crop in yields:
                    yields[crop] += int(tile.get("yield_units", 0) or 0)

            elif kind == "WEED":
                counts["WEED"] += 1

            elif kind in ("PASTURE", "COOP"):
                counts[kind] += 1
                animal = tile.get("animal")
                if animal in counts:
                    counts[animal] += 1

                product = {
                    "COW": "MILK",
                    "SHEEP": "WOOL",
                    "GOOSE": "EGG",
                }.get(animal)

                if product:
                    yields[product] += int(tile.get("yield_units", 0) or 0)

    return counts, yields


def encode_public(obs):
    player = int(obs.get("player", 0) or 0)
    farms = obs.get("farms", []) or [{}, {}]
    own = farms[player] if player < len(farms) else {}
    opp = farms[1 - player] if len(farms) > 1 else {}

    step = int(obs.get("step", 0) or 0)
    feature = [
        step / 719.0,
        (step // 24) / 29.0,
        (step % 24) / 23.0,
    ]

    for farm in (own, opp):
        feature.extend(
            [
                float(farm.get("money", 0) or 0) / 60000.0,
                len(farm.get("hands", []) or []) / 16.0,
                float(farm.get("hires_today", 0) or 0) / 16.0,
            ]
        )

        unlocked = set(farm.get("unlocked_quadrants", []) or [])
        feature.extend(
            1.0 if quadrant in unlocked else 0.0
            for quadrant in ("NW", "NE", "SW", "SE")
        )

        counts, yields = farm_summary(farm)

        feature.extend(
            counts[key] / 25.0
            for key in CROPS + ANIMALS + ["WEED", "PASTURE", "COOP"]
        )
        feature.extend(yields[key] / 100.0 for key in PRODUCTS)

    market = obs.get("market", {}) or {}
    inventory = market.get("inventory", {}) or {}
    prices = market.get("prices", {}) or {}

    for item in PRODUCTS:
        feature.append(
            (float(inventory.get(item, 10000) or 0) - 10000.0) / 500.0
        )
        feature.append(
            float(prices.get(item, BASE_PRICE[item]) or 0) / 300.0
        )

    shop_counts = Counter(
        (obs.get("town", {}) or {}).get("unlocked_shops", []) or []
    )
    feature.extend(shop_counts[name] / 8.0 for name in SHOP_NAMES)

    return np.asarray(feature, dtype=np.float32)


def encode_private(obs):
    private = obs.get("private", {}) or {}
    shed = private.get("shed", {}) or {}
    seeds = private.get("seeds", {}) or {}

    feature = [
        float(shed.get(item, 0) or 0) / 100.0
        for item in ALL_ITEMS
    ]
    feature.extend(
        float(seeds.get(crop, 0) or 0) / 50.0
        for crop in CROPS
    )

    return np.asarray(feature, dtype=np.float32)


def tile_features(farm, position):
    tiles = farm.get("tiles", []) or []

    try:
        x, y = int(position[0]), int(position[1])
    except (TypeError, ValueError, IndexError):
        x, y = 0, 0

    tile = (
        tiles[y][x]
        if 0 <= y < len(tiles) and 0 <= x < len(tiles[y])
        else None
    )

    kinds = ["NONE", "LOCKED", "WEED", "PLANT", "PASTURE", "COOP"]

    if tile is None:
        kind = "NONE"
    elif tile == "LOCKED":
        kind = "LOCKED"
    elif isinstance(tile, dict):
        kind = tile.get("kind", "NONE")
    else:
        kind = "NONE"

    feature = [1.0 if kind == key else 0.0 for key in kinds]

    crop = tile.get("crop") if isinstance(tile, dict) else None
    animal = tile.get("animal") if isinstance(tile, dict) else None

    feature.extend(1.0 if crop == value else 0.0 for value in CROPS)
    feature.extend(1.0 if animal == value else 0.0 for value in ANIMALS)

    if isinstance(tile, dict):
        feature.extend(
            [
                float(tile.get("yield_units", 0) or 0) / 10.0,
                float(bool(tile.get("watered_today", False))),
                float(bool(tile.get("fed_today", False))),
                float(bool(tile.get("cared_today", False))),
                float(bool(tile.get("fertilizer_available", False))),
                float(tile.get("consecutive_unwatered", 0) or 0) / 2.0,
                float(tile.get("consecutive_unfed", 0) or 0) / 2.0,
            ]
        )
    else:
        feature.extend([0.0] * 7)

    return feature


def encode_actor(obs, actor_index):
    player = int(obs.get("player", 0) or 0)
    farms = obs.get("farms", []) or [{}, {}]
    farm = farms[player] if player < len(farms) else {}

    private = obs.get("private", {}) or {}

    if actor_index == 0:
        position = farm.get("farmer", [0, 0])
    else:
        hands = farm.get("hands", []) or []
        position = (
            hands[actor_index - 1]
            if actor_index - 1 < len(hands)
            else [0, 0]
        )

    inventories = private.get("inventories", []) or []
    inventory = (
        inventories[actor_index]
        if actor_index < len(inventories)
        else {}
    )

    try:
        x, y = int(position[0]), int(position[1])
    except (TypeError, ValueError, IndexError):
        x, y = 0, 0

    feature = [
        1.0 if actor_index == 0 else 0.0,
        actor_index / 16.0,
        x / 9.0,
        y / 9.0,
    ]

    feature.extend(
        float(inventory.get(item, 0) or 0) / 20.0
        for item in ALL_ITEMS
    )
    feature.extend(tile_features(farm, position))

    return np.asarray(feature, dtype=np.float32)


def parse_unit(order):
    order = list(order or ["PASS"])
    op = order[0] if order else "PASS"
    item = "NONE"
    quantity = 0

    if op == "PLANT" and len(order) >= 2:
        item = str(order[1])

    elif op in ("PICKUP", "PLACE") and len(order) >= 2:
        item = str(order[1])
        quantity = int(order[2]) if len(order) >= 3 else 1

    return (
        UNIT_OP_TO_ID.get(op, UNIT_OP_TO_ID["PASS"]),
        ITEM_TO_ID.get(item, ITEM_TO_ID["NONE"]),
        min(MAX_QTY, max(0, quantity)),
    )


def parse_market(order):
    if not order:
        return (
            MARKET_OP_TO_ID["PAD"],
            ITEM_TO_ID["NONE"],
            0,
        )

    order = list(order)
    op = order[0]
    item = "NONE"
    quantity = 0

    if op in ("BUY_SEED", "BUY_PRODUCT", "BUY_ANIMAL", "SELL"):
        if len(order) >= 2:
            item = str(order[1])
        if len(order) >= 3:
            quantity = int(order[2])

    return (
        MARKET_OP_TO_ID.get(op, MARKET_OP_TO_ID["PAD"]),
        ITEM_TO_ID.get(item, ITEM_TO_ID["NONE"]),
        min(MAX_QTY, max(0, quantity)),
    )


In [ ]:

class BCDataset(Dataset):
    def __init__(self, records):
        self.records = list(records)

        dummy_obs = {
            "player": 0,
            "step": 0,
            "farms": [
                {"farmer": [0, 0], "hands": [], "tiles": []},
                {},
            ],
            "private": {},
            "market": {},
            "town": {},
        }

        reference = self.records[0]["obs"] if self.records else dummy_obs

        self.public_dim = len(encode_public(reference))
        self.private_dim = len(encode_private(reference))
        self.actor_dim = len(encode_actor(reference, 0))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        obs = row["obs"]
        action = row["action"]
        opponent_action = row["opponent_action"]

        public = encode_public(obs)
        private = encode_private(obs)

        actors = np.zeros((MAX_ACTORS, self.actor_dim), dtype=np.float32)
        actor_mask = np.zeros(MAX_ACTORS, dtype=np.float32)

        unit_op = np.full(MAX_ACTORS, -100, dtype=np.int64)
        unit_item = np.full(MAX_ACTORS, -100, dtype=np.int64)
        unit_qty = np.full(MAX_ACTORS, -100, dtype=np.int64)

        unit_orders = [
            action.get("farmer", ["PASS"]),
            *(action.get("hands", []) or []),
        ]

        for actor_index, order in enumerate(unit_orders[:MAX_ACTORS]):
            actors[actor_index] = encode_actor(obs, actor_index)
            actor_mask[actor_index] = 1.0

            op, item, qty = parse_unit(order)
            unit_op[actor_index] = op
            unit_item[actor_index] = item
            unit_qty[actor_index] = qty

        market_op = np.full(
            MAX_MARKET_ORDERS,
            MARKET_OP_TO_ID["PAD"],
            dtype=np.int64,
        )
        market_item = np.full(
            MAX_MARKET_ORDERS,
            ITEM_TO_ID["NONE"],
            dtype=np.int64,
        )
        market_qty = np.zeros(MAX_MARKET_ORDERS, dtype=np.int64)

        for slot, order in enumerate(
            (action.get("market", []) or [])[:MAX_MARKET_ORDERS]
        ):
            op, item, qty = parse_market(order)
            market_op[slot] = op
            market_item[slot] = item
            market_qty[slot] = qty

        opp_op = np.full(
            MAX_MARKET_ORDERS,
            MARKET_OP_TO_ID["PAD"],
            dtype=np.int64,
        )
        opp_item = np.full(
            MAX_MARKET_ORDERS,
            ITEM_TO_ID["NONE"],
            dtype=np.int64,
        )
        opp_qty = np.zeros(MAX_MARKET_ORDERS, dtype=np.int64)

        for slot, order in enumerate(
            (opponent_action.get("market", []) or [])[:MAX_MARKET_ORDERS]
        ):
            op, item, qty = parse_market(order)
            opp_op[slot] = op
            opp_item[slot] = item
            opp_qty[slot] = qty

        return {
            "public": torch.from_numpy(public),
            "private": torch.from_numpy(private),
            "actors": torch.from_numpy(actors),
            "actor_mask": torch.from_numpy(actor_mask),
            "unit_op": torch.from_numpy(unit_op),
            "unit_item": torch.from_numpy(unit_item),
            "unit_qty": torch.from_numpy(unit_qty),
            "market_op": torch.from_numpy(market_op),
            "market_item": torch.from_numpy(market_item),
            "market_qty": torch.from_numpy(market_qty),
            "opp_market_op": torch.from_numpy(opp_op),
            "opp_market_item": torch.from_numpy(opp_item),
            "opp_market_qty": torch.from_numpy(opp_qty),
            "win": torch.tensor(row["win"], dtype=torch.float32),
            "margin": torch.tensor(row["margin"], dtype=torch.float32),
            "next_day_income": torch.tensor(
                row["next_day_income"],
                dtype=torch.float32,
            ),
        }


train_dataset = BCDataset(train_records)
val_dataset = BCDataset(val_records)

print(
    "feature dims:",
    train_dataset.public_dim,
    train_dataset.private_dim,
    train_dataset.actor_dim,
)


In [ ]:

class PolicyNet(nn.Module):
    def __init__(self, public_dim, private_dim, actor_dim, width=256):
        super().__init__()

        self.step_embedding = nn.Embedding(720, 64)
        self.actor_embedding = nn.Embedding(MAX_ACTORS, 32)
        self.slot_embedding = nn.Embedding(MAX_MARKET_ORDERS, 32)

        self.public_encoder = nn.Sequential(
            nn.Linear(public_dim, width),
            nn.GELU(),
            nn.Linear(width, width),
            nn.GELU(),
        )

        self.private_encoder = nn.Sequential(
            nn.Linear(private_dim, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
        )

        core_dim = width + 128 + 64
        public_core_dim = width + 64

        unit_dim = core_dim + actor_dim + 32
        market_dim = core_dim + 32
        opponent_dim = public_core_dim + 32

        self.unit_op_head = nn.Sequential(
            nn.Linear(unit_dim, width),
            nn.GELU(),
            nn.Linear(width, len(UNIT_OPS)),
        )
        self.unit_item_head = nn.Sequential(
            nn.Linear(unit_dim, 192),
            nn.GELU(),
            nn.Linear(192, len(ITEMS)),
        )
        self.unit_qty_head = nn.Sequential(
            nn.Linear(unit_dim, 192),
            nn.GELU(),
            nn.Linear(192, MAX_QTY + 1),
        )

        self.market_op_head = nn.Sequential(
            nn.Linear(market_dim, width),
            nn.GELU(),
            nn.Linear(width, len(MARKET_OPS)),
        )
        self.market_item_head = nn.Sequential(
            nn.Linear(market_dim, 192),
            nn.GELU(),
            nn.Linear(192, len(ITEMS)),
        )
        self.market_qty_head = nn.Sequential(
            nn.Linear(market_dim, 192),
            nn.GELU(),
            nn.Linear(192, MAX_QTY + 1),
        )

        # Public-only opponent prediction head.
        self.opp_market_op_head = nn.Sequential(
            nn.Linear(opponent_dim, 192),
            nn.GELU(),
            nn.Linear(192, len(MARKET_OPS)),
        )
        self.opp_market_item_head = nn.Sequential(
            nn.Linear(opponent_dim, 160),
            nn.GELU(),
            nn.Linear(160, len(ITEMS)),
        )
        self.opp_market_qty_head = nn.Sequential(
            nn.Linear(opponent_dim, 160),
            nn.GELU(),
            nn.Linear(160, MAX_QTY + 1),
        )

        # win logit, final margin, next-24-step income
        self.value_head = nn.Sequential(
            nn.Linear(core_dim, 192),
            nn.GELU(),
            nn.Linear(192, 3),
        )

    def forward(self, public, private, actors):
        batch_size = public.shape[0]

        step = (
            public[:, 0]
            .mul(719)
            .round()
            .long()
            .clamp(0, 719)
        )

        step_context = self.step_embedding(step)
        public_context = self.public_encoder(public)
        private_context = self.private_encoder(private)

        core = torch.cat(
            [public_context, private_context, step_context],
            dim=-1,
        )
        public_core = torch.cat(
            [public_context, step_context],
            dim=-1,
        )

        actor_ids = torch.arange(
            actors.shape[1],
            device=actors.device,
        )
        actor_embedding = (
            self.actor_embedding(actor_ids)
            .unsqueeze(0)
            .expand(batch_size, -1, -1)
        )

        unit_core = core.unsqueeze(1).expand(
            -1,
            actors.shape[1],
            -1,
        )
        unit_input = torch.cat(
            [unit_core, actors, actor_embedding],
            dim=-1,
        )

        slot_ids = torch.arange(
            MAX_MARKET_ORDERS,
            device=public.device,
        )
        slot_embedding = (
            self.slot_embedding(slot_ids)
            .unsqueeze(0)
            .expand(batch_size, -1, -1)
        )

        market_core = core.unsqueeze(1).expand(
            -1,
            MAX_MARKET_ORDERS,
            -1,
        )
        market_input = torch.cat(
            [market_core, slot_embedding],
            dim=-1,
        )

        opponent_core = public_core.unsqueeze(1).expand(
            -1,
            MAX_MARKET_ORDERS,
            -1,
        )
        opponent_input = torch.cat(
            [opponent_core, slot_embedding],
            dim=-1,
        )

        return {
            "unit_op": self.unit_op_head(unit_input),
            "unit_item": self.unit_item_head(unit_input),
            "unit_qty": self.unit_qty_head(unit_input),
            "market_op": self.market_op_head(market_input),
            "market_item": self.market_item_head(market_input),
            "market_qty": self.market_qty_head(market_input),
            "opp_market_op": self.opp_market_op_head(opponent_input),
            "opp_market_item": self.opp_market_item_head(opponent_input),
            "opp_market_qty": self.opp_market_qty_head(opponent_input),
            "value": self.value_head(core),
        }


model = PolicyNet(
    train_dataset.public_dim,
    train_dataset.private_dim,
    train_dataset.actor_dim,
).to(DEVICE)

# Important: no weight decay on embeddings, biases, or 1-D parameters.
decay_params = []
no_decay_params = []

for name, parameter in model.named_parameters():
    if (
        parameter.ndim < 2
        or "embedding" in name
        or name.endswith(".bias")
    ):
        no_decay_params.append(parameter)
    else:
        decay_params.append(parameter)

optimizer = torch.optim.AdamW(
    [
        {
            "params": decay_params,
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": no_decay_params,
            "weight_decay": 0.0,
        },
    ],
    lr=LR,
)

print(
    "parameters:",
    sum(parameter.numel() for parameter in model.parameters()),
)


In [ ]:

def conditional_ce(logits, targets, mask):
    if mask.sum().item() == 0:
        return logits.sum() * 0.0

    return F.cross_entropy(
        logits[mask],
        targets[mask],
    )


def compute_loss(batch):
    batch = {
        key: value.to(DEVICE)
        for key, value in batch.items()
    }

    output = model(
        batch["public"].float(),
        batch["private"].float(),
        batch["actors"].float(),
    )

    actor_mask = batch["actor_mask"].bool()

    unit_op_loss = F.cross_entropy(
        output["unit_op"][actor_mask],
        batch["unit_op"][actor_mask],
    )

    unit_item_mask = actor_mask & (
        (batch["unit_op"] == UNIT_OP_TO_ID["PLANT"])
        | (batch["unit_op"] == UNIT_OP_TO_ID["PICKUP"])
        | (batch["unit_op"] == UNIT_OP_TO_ID["PLACE"])
    )

    unit_qty_mask = actor_mask & (
        (batch["unit_op"] == UNIT_OP_TO_ID["PICKUP"])
        | (batch["unit_op"] == UNIT_OP_TO_ID["PLACE"])
    )

    unit_item_loss = conditional_ce(
        output["unit_item"],
        batch["unit_item"],
        unit_item_mask,
    )
    unit_qty_loss = conditional_ce(
        output["unit_qty"],
        batch["unit_qty"],
        unit_qty_mask,
    )

    market_op_weight = torch.ones(
        len(MARKET_OPS),
        device=DEVICE,
    )
    market_op_weight[MARKET_OP_TO_ID["PAD"]] = 0.25

    market_op_loss = F.cross_entropy(
        output["market_op"].reshape(-1, len(MARKET_OPS)),
        batch["market_op"].reshape(-1),
        weight=market_op_weight,
    )

    market_arg_mask = (
        (batch["market_op"] == MARKET_OP_TO_ID["BUY_SEED"])
        | (batch["market_op"] == MARKET_OP_TO_ID["BUY_PRODUCT"])
        | (batch["market_op"] == MARKET_OP_TO_ID["BUY_ANIMAL"])
        | (batch["market_op"] == MARKET_OP_TO_ID["SELL"])
    )

    market_item_loss = conditional_ce(
        output["market_item"],
        batch["market_item"],
        market_arg_mask,
    )
    market_qty_loss = conditional_ce(
        output["market_qty"],
        batch["market_qty"],
        market_arg_mask,
    )

    opponent_op_loss = F.cross_entropy(
        output["opp_market_op"].reshape(-1, len(MARKET_OPS)),
        batch["opp_market_op"].reshape(-1),
        weight=market_op_weight,
    )

    opponent_arg_mask = (
        (batch["opp_market_op"] == MARKET_OP_TO_ID["BUY_SEED"])
        | (batch["opp_market_op"] == MARKET_OP_TO_ID["BUY_PRODUCT"])
        | (batch["opp_market_op"] == MARKET_OP_TO_ID["BUY_ANIMAL"])
        | (batch["opp_market_op"] == MARKET_OP_TO_ID["SELL"])
    )

    opponent_item_loss = conditional_ce(
        output["opp_market_item"],
        batch["opp_market_item"],
        opponent_arg_mask,
    )
    opponent_qty_loss = conditional_ce(
        output["opp_market_qty"],
        batch["opp_market_qty"],
        opponent_arg_mask,
    )

    value = output["value"]

    win_loss = F.binary_cross_entropy_with_logits(
        value[:, 0],
        batch["win"].float(),
    )
    margin_loss = F.smooth_l1_loss(
        value[:, 1],
        batch["margin"].float(),
    )
    next_day_loss = F.smooth_l1_loss(
        value[:, 2],
        batch["next_day_income"].float(),
    )

    policy_loss = (
        unit_op_loss
        + 0.50 * unit_item_loss
        + 0.35 * unit_qty_loss
        + market_op_loss
        + 0.50 * market_item_loss
        + 0.35 * market_qty_loss
    )

    auxiliary_loss = (
        0.20 * opponent_op_loss
        + 0.10 * opponent_item_loss
        + 0.08 * opponent_qty_loss
        + 0.20 * win_loss
        + 0.08 * margin_loss
        + 0.08 * next_day_loss
    )

    return policy_loss + auxiliary_loss


def component_metrics(dataset):
    if len(dataset) == 0:
        return {}

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    model.eval()
    total = Counter()
    correct = Counter()

    with torch.no_grad():
        for batch in loader:
            batch = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }

            output = model(
                batch["public"].float(),
                batch["private"].float(),
                batch["actors"].float(),
            )

            actor_mask = batch["actor_mask"].bool()

            unit_op_pred = output["unit_op"].argmax(dim=-1)
            unit_item_pred = output["unit_item"].argmax(dim=-1)
            unit_qty_pred = output["unit_qty"].argmax(dim=-1)

            market_op_pred = output["market_op"].argmax(dim=-1)
            market_item_pred = output["market_item"].argmax(dim=-1)
            market_qty_pred = output["market_qty"].argmax(dim=-1)

            unit_exact = (
                (unit_op_pred == batch["unit_op"])
                & actor_mask
            )

            item_needed = (
                (batch["unit_op"] == UNIT_OP_TO_ID["PLANT"])
                | (batch["unit_op"] == UNIT_OP_TO_ID["PICKUP"])
                | (batch["unit_op"] == UNIT_OP_TO_ID["PLACE"])
            )
            qty_needed = (
                (batch["unit_op"] == UNIT_OP_TO_ID["PICKUP"])
                | (batch["unit_op"] == UNIT_OP_TO_ID["PLACE"])
            )

            unit_exact &= (
                (~item_needed)
                | (unit_item_pred == batch["unit_item"])
            )
            unit_exact &= (
                (~qty_needed)
                | (unit_qty_pred == batch["unit_qty"])
            )

            market_exact = (
                market_op_pred == batch["market_op"]
            )

            market_args_needed = (
                (batch["market_op"] == MARKET_OP_TO_ID["BUY_SEED"])
                | (batch["market_op"] == MARKET_OP_TO_ID["BUY_PRODUCT"])
                | (batch["market_op"] == MARKET_OP_TO_ID["BUY_ANIMAL"])
                | (batch["market_op"] == MARKET_OP_TO_ID["SELL"])
            )

            market_exact &= (
                (~market_args_needed)
                | (market_item_pred == batch["market_item"])
            )
            market_exact &= (
                (~market_args_needed)
                | (market_qty_pred == batch["market_qty"])
            )

            full_exact = (
                (unit_exact | (~actor_mask)).all(dim=1)
                & market_exact.all(dim=1)
            )

            correct["unit"] += unit_exact.sum().item()
            total["unit"] += actor_mask.sum().item()

            correct["market_slot"] += market_exact.sum().item()
            total["market_slot"] += market_exact.numel()

            correct["full"] += full_exact.sum().item()
            total["full"] += len(full_exact)

    return {
        name: correct[name] / max(1, total[name])
        for name in ("unit", "market_slot", "full")
    }


def train_epochs(dataset, epochs):
    if len(dataset) == 0:
        print("No collected rollout data. Training skipped.")
        return

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        batches = 0

        for batch in loader:
            optimizer.zero_grad(set_to_none=True)

            loss = compute_loss(batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
            )
            optimizer.step()

            running_loss += float(loss.detach().cpu())
            batches += 1

        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(
                f"epoch={epoch:03d}",
                f"loss={running_loss / max(1, batches):.4f}",
                "train=", component_metrics(dataset),
                "val=", component_metrics(val_dataset),
            )


train_epochs(train_dataset, BC_EPOCHS)

torch.save(
    model.state_dict(),
    "bc_before_dagger.pt",
)

bc_metrics = component_metrics(val_dataset)
print("BC validation:", bc_metrics)


In [ ]:

UNIT_NO_ARG = {
    "PASS", "NORTH", "SOUTH", "EAST", "WEST",
    "DROP", "WATER", "HARVEST", "FERTILIZE",
    "BUILD_COOP", "BUILD_PASTURE", "DIG",
    "FEED", "COLLECT_FERTILIZER", "CARE",
}


def masked_item_argmax(logits, allowed):
    ids = [ITEM_TO_ID[value] for value in allowed]
    values = logits[ids]
    return ids[int(values.argmax().item())]


def decode_unit(op_id, item_logits, qty_logits):
    op = ID_TO_UNIT_OP[int(op_id)]

    if op in UNIT_NO_ARG:
        return [op]

    if op == "PLANT":
        item_id = masked_item_argmax(item_logits, CROPS)
        return ["PLANT", ID_TO_ITEM[item_id]]

    if op in ("PICKUP", "PLACE"):
        item_id = masked_item_argmax(item_logits, ALL_ITEMS)
        quantity = max(1, int(qty_logits.argmax().item()))
        return [op, ID_TO_ITEM[item_id], quantity]

    return ["PASS"]


def decode_market(op_id, item_logits, qty_logits):
    op = ID_TO_MARKET_OP[int(op_id)]

    if op == "PAD":
        return None

    if op in ("HIRE", "BUY_LAND"):
        return [op]

    if op == "BUY_SEED":
        allowed = CROPS
    elif op == "BUY_PRODUCT":
        allowed = ["WHEAT", "FERTILIZER"]
    elif op == "BUY_ANIMAL":
        allowed = ANIMALS
    elif op == "SELL":
        allowed = PRODUCTS
    else:
        return None

    item_id = masked_item_argmax(
        item_logits,
        allowed,
    )
    quantity = max(
        1,
        int(qty_logits.argmax().item()),
    )

    return [op, ID_TO_ITEM[item_id], quantity]


def sanitize_action(obs, action):
    player = int(obs.get("player", 0) or 0)
    farms = obs.get("farms", []) or [{}, {}]
    farm = farms[player] if player < len(farms) else {}
    private = obs.get("private", {}) or {}

    hand_count = len(farm.get("hands", []) or [])

    action["hands"] = list(
        action.get("hands", []) or []
    )[:hand_count]

    while len(action["hands"]) < hand_count:
        action["hands"].append(["PASS"])

    shed = dict(private.get("shed", {}) or {})
    inventories = list(private.get("inventories", []) or [])

    unit_orders = [
        action.get("farmer", ["PASS"]),
        *action["hands"],
    ]

    for actor_index, order in enumerate(unit_orders):
        if not order:
            continue

        if order[0] == "PLACE" and len(order) >= 3:
            inventory = (
                inventories[actor_index]
                if actor_index < len(inventories)
                else {}
            )

            available = max(
                0,
                int(inventory.get(order[1], 0) or 0),
            )
            if available <= 0:
                order[:] = ["PASS"]
            else:
                order[2] = min(
                    max(1, int(order[2])),
                    available,
                )

        elif order[0] == "PICKUP" and len(order) >= 3:
            available = max(
                0,
                int(shed.get(order[1], 0) or 0),
            )
            if available <= 0:
                order[:] = ["PASS"]
            else:
                order[2] = min(
                    max(1, int(order[2])),
                    available,
                )

    remaining = {
        item: max(0, int(shed.get(item, 0) or 0))
        for item in PRODUCTS
    }

    clean_market = []

    for order in action.get("market", []) or []:
        if len(clean_market) >= MAX_MARKET_ORDERS:
            break

        if not order:
            continue

        order = list(order)

        if order[0] == "SELL" and len(order) >= 3:
            item = order[1]
            quantity = min(
                max(1, int(order[2])),
                remaining.get(item, 0),
            )

            if quantity <= 0:
                continue

            order[2] = quantity
            remaining[item] -= quantity

        clean_market.append(order)

    action["market"] = clean_market
    return action


def student_action(obs):
    model.eval()

    player = int(obs.get("player", 0) or 0)
    farms = obs.get("farms", []) or [{}, {}]
    farm = farms[player] if player < len(farms) else {}

    actor_count = min(
        MAX_ACTORS,
        1 + len(farm.get("hands", []) or []),
    )

    public = torch.from_numpy(
        encode_public(obs)
    ).unsqueeze(0).to(DEVICE)

    private = torch.from_numpy(
        encode_private(obs)
    ).unsqueeze(0).to(DEVICE)

    actors_np = np.zeros(
        (1, MAX_ACTORS, train_dataset.actor_dim),
        dtype=np.float32,
    )

    for actor_index in range(actor_count):
        actors_np[0, actor_index] = encode_actor(
            obs,
            actor_index,
        )

    actors = torch.from_numpy(
        actors_np
    ).to(DEVICE)

    with torch.no_grad():
        output = model(
            public.float(),
            private.float(),
            actors.float(),
        )

    unit_orders = []

    for actor_index in range(actor_count):
        op_id = output["unit_op"][
            0,
            actor_index,
        ].argmax().item()

        unit_orders.append(
            decode_unit(
                op_id,
                output["unit_item"][0, actor_index],
                output["unit_qty"][0, actor_index],
            )
        )

    market_orders = []

    for slot in range(MAX_MARKET_ORDERS):
        op_id = output["market_op"][
            0,
            slot,
        ].argmax().item()

        order = decode_market(
            op_id,
            output["market_item"][0, slot],
            output["market_qty"][0, slot],
        )

        if order is not None:
            market_orders.append(order)

    action = {
        "farmer": unit_orders[0] if unit_orders else ["PASS"],
        "hands": unit_orders[1:],
        "market": market_orders[:MAX_MARKET_ORDERS],
    }

    return sanitize_action(obs, action)


def fresh_student():
    def policy(obs):
        return student_action(to_plain(obs))

    return policy


In [ ]:

def collect_dagger_episode(seed, opponent_kind, student_seat):
    if not HAVE_ENV:
        return []

    oracle = make_teacher()
    queried = []

    def wrapped_student(raw_obs):
        obs = to_plain(raw_obs)
        step = int(obs.get("step", 0) or 0)

        teacher_label = oracle(copy.deepcopy(obs))
        predicted = student_action(obs)

        queried.append(
            {
                "step": step,
                "obs": obs,
                "action": teacher_label,
            }
        )

        return predicted

    if opponent_kind == "teacher":
        opponent = make_teacher()
    elif opponent_kind == "random":
        opponent = "random"
    else:
        raise ValueError(opponent_kind)

    agents = [None, None]
    agents[student_seat] = wrapped_student
    agents[1 - student_seat] = opponent

    env = make(
        "kaggriculture",
        configuration={"episodeSteps": 720, "seed": int(seed)},
        debug=False,
    )
    env.run(agents)

    replay = env.toJSON()
    final_rewards = replay.get("rewards") or [0, 0]
    opponent_seat = 1 - student_seat

    records = []

    for row in queried:
        step = row["step"]
        obs = row["obs"]

        next_index = min(
            step + 1,
            len(replay["steps"]) - 1,
        )

        opponent_action = copy.deepcopy(
            replay["steps"][next_index][opponent_seat]["action"]
        )

        current_money = float(
            (obs.get("farms") or [{}, {}])[student_seat].get("money", 0) or 0
        )

        future_index = min(
            len(replay["steps"]) - 1,
            step + 24,
        )
        future_obs = replay["steps"][future_index][student_seat]["observation"]

        future_money = float(
            (future_obs.get("farms") or [{}, {}])[student_seat].get("money", 0) or 0
        )

        records.append(
            {
                "episode_id": f"dagger-{seed}-{student_seat}-{opponent_kind}",
                "seat": student_seat,
                "step": step,
                "obs": obs,
                "action": row["action"],
                "opponent_action": opponent_action,
                "win": float(
                    final_rewards[student_seat] > final_rewards[opponent_seat]
                ),
                "margin": (
                    final_rewards[student_seat]
                    - final_rewards[opponent_seat]
                ) / 10000.0,
                "next_day_income": (
                    future_money - current_money
                ) / 10000.0,
            }
        )

    print(
        "DAgger",
        opponent_kind,
        f"seat={student_seat}",
        f"reward={final_rewards}",
        f"queries={len(records)}",
    )

    return records


dagger_records = []

if HAVE_ENV and len(train_records):
    for game_index in range(DAGGER_GAMES):
        dagger_records.extend(
            collect_dagger_episode(
                seed=15000 + game_index,
                opponent_kind=(
                    "teacher"
                    if game_index % 2 == 0
                    else "random"
                ),
                student_seat=game_index % 2,
            )
        )

if dagger_records:
    aggregated_records = train_records + dagger_records
    aggregated_dataset = BCDataset(aggregated_records)

    train_epochs(
        aggregated_dataset,
        DAGGER_EPOCHS,
    )
else:
    aggregated_dataset = train_dataset

torch.save(
    model.state_dict(),
    "bc_after_dagger.pt",
)

dagger_metrics = component_metrics(val_dataset)
print("post-DAgger validation:", dagger_metrics)


In [ ]:

def play_match(agent_a, agent_b, seed):
    env = make(
        "kaggriculture",
        configuration={"episodeSteps": 720, "seed": int(seed)},
        debug=False,
    )
    env.run([agent_a, agent_b])

    replay = env.toJSON()
    rewards = replay.get("rewards") or [0, 0]

    return float(rewards[0]), float(rewards[1])


def evaluate_student(num_games=4):
    if not HAVE_ENV or len(train_records) == 0:
        return {}

    opponent_factories = {
        "teacher": make_teacher,
        "random": lambda: "random",
    }

    report = {}

    for name, factory in opponent_factories.items():
        wins = 0
        margins = []

        for game_index in range(num_games):
            seed = 20000 + game_index
            student = fresh_student()
            opponent = factory()

            if game_index % 2 == 0:
                left, right = play_match(
                    student,
                    opponent,
                    seed,
                )
                student_score = left
                opponent_score = right
            else:
                left, right = play_match(
                    opponent,
                    student,
                    seed,
                )
                student_score = right
                opponent_score = left

            wins += int(student_score > opponent_score)
            margins.append(
                student_score - opponent_score
            )

        report[name] = {
            "win_rate": wins / num_games,
            "mean_margin": float(np.mean(margins)),
        }

    return report


rollout_report = evaluate_student(
    num_games=2 if FAST_MODE else 10
)

final_metrics = component_metrics(val_dataset)

gate = (
    final_metrics.get("full", 0.0) >= BC_GATE_FULL_ACTION
    and final_metrics.get("unit", 0.0) >= BC_GATE_UNIT
    and final_metrics.get("market_slot", 0.0) >= BC_GATE_MARKET_SLOT
)

print("rollout report:")
print(json.dumps(rollout_report, indent=2))

print("final BC metrics:", final_metrics)
print("BC GATE:", "PASS" if gate else "FAIL")

if not gate:
    print("STOP: improve imitation data / representation before PPO.")
else:
    print("BC is strong enough to begin population-response experiments.")


In [ ]:

def empirical_payoff(policy_factories, seeds):
    """Win/loss payoff matrix; coin margin is not the optimization target."""
    n = len(policy_factories)
    matrix = np.zeros((n, n), dtype=np.float64)

    for i in range(n):
        for j in range(i + 1, n):
            outcomes = []

            for game_index, seed in enumerate(seeds):
                left_policy = policy_factories[i]()
                right_policy = policy_factories[j]()

                if game_index % 2 == 0:
                    left_score, right_score = play_match(
                        left_policy,
                        right_policy,
                        seed,
                    )
                    outcome = np.sign(
                        left_score - right_score
                    )
                else:
                    right_score, left_score = play_match(
                        right_policy,
                        left_policy,
                        seed,
                    )
                    outcome = np.sign(
                        left_score - right_score
                    )

                outcomes.append(outcome)

            value = float(np.mean(outcomes))
            matrix[i, j] = value
            matrix[j, i] = -value

    return matrix


def multiplicative_weights_meta(
    payoff,
    iterations=4000,
    eta=0.05,
):
    n = payoff.shape[0]

    row = np.ones(n, dtype=np.float64) / n
    col = np.ones(n, dtype=np.float64) / n

    row_average = np.zeros(n, dtype=np.float64)
    col_average = np.zeros(n, dtype=np.float64)

    for _ in range(iterations):
        row_value = payoff @ col
        col_value = row @ payoff

        row *= np.exp(eta * row_value)
        col *= np.exp(-eta * col_value)

        row /= row.sum()
        col /= col.sum()

        row_average += row
        col_average += col

    return (
        row_average / iterations,
        col_average / iterations,
    )


if HAVE_ENV and gate:
    population_names = [
        "teacher",
        "bc_dagger",
    ]
    population = [
        make_teacher,
        fresh_student,
    ]

    payoff = empirical_payoff(
        population,
        seeds=list(
            range(
                30000,
                30004 if FAST_MODE else 30020,
            )
        ),
    )

    row_mix, col_mix = multiplicative_weights_meta(
        payoff
    )

    print("population:", population_names)
    print("payoff:")
    print(payoff)
    print("row mixture:", row_mix)
    print("column mixture:", col_mix)

else:
    payoff = None
    row_mix = None
    col_mix = None
    print("PSRO stage skipped until the BC gate passes.")


In [ ]:

summary = {
    "teacher_sha256": 'df4e899ad535754cf2ddbd3c16e48085916b0cd2baa5182a1a2cfc6a856abae5',
    "bc_metrics": bc_metrics,
    "post_dagger_metrics": dagger_metrics,
    "final_metrics": final_metrics,
    "rollout_report": rollout_report,
    "bc_gate": bool(gate),
}

Path("pipeline_metrics.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print("saved: bc_before_dagger.pt")
print("saved: bc_after_dagger.pt")
print("saved: pipeline_metrics.json")


## Promotion rules

Do **not** add PPO while the BC gate is red.

After the gate passes:

- add many independent high-rated public replays;
- keep train/validation split at the episode level;
- scale the BC network wider and verify cloning still improves;
- add a one-turn exact market-response search using the public opponent head;
- expand the PSRO population with frozen checkpoints and strong public policies;
- only then warm-start PPO from the BC/DAgger checkpoint against the PSRO mixture;
- evaluate every candidate on win rate over a held-out seed/opponent set before leaderboard submission.


In [ ]:
# Export a safe leaderboard artifact.
# Until the learned policy passes the BC gate, submission.py stays equal to
# the frozen teacher. This prevents a failed training run from producing a
# weaker leaderboard agent.

from pathlib import Path

teacher_source = Path("teacher_agent.py").read_text(encoding="utf-8")

submission_target = (
    Path("/kaggle/working/submission.py")
    if Path("/kaggle/working").exists()
    else Path("submission.py")
)

submission_target.write_text(teacher_source, encoding="utf-8")
compile(submission_target.read_text(encoding="utf-8"), str(submission_target), "exec")

print("submission.py:", submission_target)
print("sha256:", "df4e899ad535754cf2ddbd3c16e48085916b0cd2baa5182a1a2cfc6a856abae5")
print("policy: frozen teacher fallback")
